In [1]:
# Combine the original, derived, trend, market, macro, and matching tables.
# Preserve one row per event and identifiable feature groups.

In [2]:
# Locate shared project configuration and reusable helpers.
import sys as _sys
from pathlib import Path as _Path

_UTILS_ROOT = next(
    path
    for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
    if (path / "notebook_utils.py").is_file()
)
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths
from project_config import (
    ORIGINAL_COLUMNS as original_columns,
    RATIO_COLUMNS as ratio_columns,
    TREND_FEATURE_COLUMNS as trend_columns,
    MARKET_FEATURE_COLUMNS as market_feature_columns,
    COMPACT_FEATURE_COLUMNS,
)
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/original_features.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
KEYS = ["company_id", "prediction_date"]
CORE = [
    "company_id",
    "prediction_date",
    "target",
    "bankruptcy_date",
    "event_type",
    "sector",
    "industry",
    "prediction_year",
]
SOURCES = [
    "original_features",
    "derived_features",
    "trend_features",
    "market_features",
    "macro_features",
]
MATCH_RATIO = 2
MATCH_SOURCE = f"matched_event_sample_1to{MATCH_RATIO}.csv"


# Explicit string IDs and daily dates make company-event keys stable across CSVs.
def load_feature_table(path):
    frame = pd.read_csv(path, dtype={"company_id": "string", "event_id": "string"})
    required = KEYS + ["event_id", "target", "prediction_year"]
    absent = sorted(set(required) - set(frame.columns))
    if absent:
        raise ValueError(f"{path.name}: missing required columns {absent}")
    if frame.empty or frame[required].isna().any().any():
        raise ValueError(f"{path.name}: missing events or keys/labels")
    ids = frame["company_id"]
    if not ids.eq(ids.str.strip()).all() or ids.eq("").any():
        raise ValueError(f"{path.name}: unstandardized company identifiers")
    date_values = {}
    for column in ["prediction_date", "bankruptcy_date"]:
        if column in frame:
            parsed = pd.to_datetime(frame[column], errors="raise").astype(
                "datetime64[ns]"
            )
            if (
                parsed.notna().any()
                and not parsed.loc[parsed.notna()]
                .eq(parsed.loc[parsed.notna()].dt.normalize())
                .all()
            ):
                raise ValueError(f"{path.name}: time components in daily {column}")
            date_values[column] = parsed
    frame = pd.DataFrame(
        {c: date_values.get(c, frame[c]) for c in frame}, index=frame.index
    )
    if frame.duplicated(KEYS).any() or not frame["event_id"].is_unique:
        raise ValueError(f"{path.name}: duplicate company-event keys")
    if (
        not frame["target"].isin([0, 1]).all()
        or not frame["prediction_year"].eq(frame["prediction_date"].dt.year).all()
    ):
        raise ValueError(f"{path.name}: invalid target or prediction year")
    return frame


# A validated one-to-one join retains the original cohort/order and adds only new columns.
def merge_feature_table(current, incoming, source_name):
    left = current.set_index(KEYS)
    right = incoming.set_index(KEYS)
    if not left.index.is_unique or not right.index.is_unique:
        raise ValueError(f"{source_name}: duplicate event keys")
    if len(left.index.difference(right.index)) or len(
        right.index.difference(left.index)
    ):
        raise ValueError(
            f"{source_name}: company-event cohort differs; rerun the affected upstream notebook"
        )
    right = right.reindex(left.index)
    new_columns = [c for c in incoming if c not in current]
    merged = current.merge(
        incoming[KEYS + new_columns],
        on=KEYS,
        how="left",
        sort=False,
        validate="one_to_one",
    )
    if len(merged) != len(current) or not merged[KEYS].equals(current[KEYS]):
        raise ValueError(f"{source_name}: join changed event identity or order")
    return merged, new_columns


tables = {source: load_feature_table(PROCESSED / f"{source}.csv") for source in SOURCES}
combined = tables[SOURCES[0]].copy()
added_columns = {SOURCES[0]: combined.columns.tolist()}
for source in SOURCES[1:]:
    combined, added_columns[source] = merge_feature_table(
        combined, tables[source], source
    )
if not set(CORE).issubset(combined.columns):
    raise ValueError("Missing core event metadata")

In [3]:
# Assign groups by source/explicit definitions, rather than a name containing 'market'.
# For example, original_market_cap, derived_market_to_book, and trend_market_cap_growth_1y
# belong to different groups. Reference dates/counts never become numeric predictors.
required_features = (
    original_columns + ratio_columns + trend_columns + market_feature_columns
)
missing_features = sorted(set(required_features) - set(combined.columns))
if missing_features:
    raise ValueError(f"Missing expected feature columns: {missing_features}")
macro_added = added_columns["macro_features"]
macro_feature_columns = [
    c for c in macro_added if "_macro_" not in c and c != "macro_feature_reference_date"
]
if not macro_feature_columns:
    raise ValueError("No macro features were contributed")
original_history_columns = [
    c
    for c in added_columns["original_features"]
    if c.startswith("market_monthly_price_")
    and not c.startswith("market_monthly_price_date_")
]
compact_original_columns = ["log_total_assets", "log_market_cap"] + [
    f"signed_log_{column}"
    for column in [
        "working_capital",
        "ebit",
        "ebitda",
        "net_income",
        "interest_expense",
        "operating_cash_flow",
        "capital_expenditures",
        "free_cash_flow",
    ]
]
missing_columns = [f"{c}_missing" for c in original_columns]
if not set(missing_columns).issubset(combined.columns):
    raise ValueError("Original missingness indicators are absent")
feature_groups = {
    "original": original_columns + compact_original_columns + original_history_columns,
    "derived": ratio_columns,
    "trend": trend_columns,
    "market": market_feature_columns,
    "macro": macro_feature_columns,
    "missing": missing_columns,
}
assigned = [c for columns in feature_groups.values() for c in columns]
if len(assigned) != len(set(assigned)) or set(assigned).intersection(CORE):
    raise ValueError("Ambiguous feature-group assignment")
feature_groups["metadata"] = [
    c for c in combined if c not in CORE and c not in assigned
]
source_of = {
    column: source for source, columns in added_columns.items() for column in columns
}
column_group = {
    column: group for group, columns in feature_groups.items() for column in columns
}
rename_map = {
    column: (
        f"missing_{column.removesuffix('_missing')}"
        if group == "missing"
        else f"{group}_{column}"
    )
    for column, group in column_group.items()
}
if len(set(rename_map.values()).union(CORE)) != len(combined.columns):
    raise ValueError("Prefixing caused column-name collisions")
column_lineage = pd.DataFrame(
    {
        "source_file": [source_of[c] + ".csv" for c in combined],
        "source_column": combined.columns,
        "output_column": [rename_map.get(c, c) for c in combined],
        "group": [column_group.get(c, "core") for c in combined],
    }
)
modeling_dataset = combined.rename(columns=rename_map)
missing_compact_columns = sorted(
    set(COMPACT_FEATURE_COLUMNS) - set(modeling_dataset.columns)
)
if missing_compact_columns:
    raise ValueError(
        f"Compact feature columns are unavailable: {missing_compact_columns}"
    )

In [4]:
# Attach all original matching links to their retained members without row expansion.
matching = pd.read_csv(
    PROCESSED / MATCH_SOURCE,
    dtype={"bankrupt_company_id": "string", "control_company_id": "string"},
)
match_fields = [
    "bankrupt_company_id",
    "control_company_id",
    "bankrupt_prediction_date",
    "control_pseudo_event_date",
    "prediction_year",
    "bankrupt_sector",
    "control_sector",
    "bankrupt_industry",
    "control_industry",
    "bankrupt_size",
    "control_size",
    "size_distance",
    "match_stage",
    "size_basis",
    "bankrupt_size_date",
    "control_size_date",
    "control_prediction_year",
    "match_rank",
    "sector_match",
    "industry_match",
    "year_match",
    "control_reuse_count",
]
if matching.empty or not set(match_fields).issubset(matching.columns):
    raise ValueError("Missing primary matching metadata")
if matching[match_fields].isna().any().any():
    raise ValueError("Incomplete required matching fields")
for column in ["bankrupt_company_id", "control_company_id"]:
    if (
        not matching[column].eq(matching[column].str.strip()).all()
        or matching[column].eq("").any()
    ):
        raise ValueError(f"Unstandardized matching identifiers in {column}")
match_dates = [
    "bankrupt_prediction_date",
    "control_pseudo_event_date",
    "bankrupt_size_date",
    "control_size_date",
]
matching = pd.DataFrame(
    {
        c: pd.to_datetime(matching[c], errors="raise").astype("datetime64[ns]")
        if c in match_dates
        else matching[c]
        for c in matching
    }
)
pair_keys = [
    "bankrupt_company_id",
    "bankrupt_prediction_date",
    "control_company_id",
    "control_pseudo_event_date",
]
if matching.duplicated(pair_keys).any() or matching.duplicated(pair_keys[:3]).any():
    raise ValueError("Repeated control company within a bankrupt event")
if (
    not matching["match_stage"].isin([1, 2, 3, 4]).all()
    or not matching["match_rank"].isin(range(1, MATCH_RATIO + 1)).all()
):
    raise ValueError("Invalid matching stage or rank")
if matching.duplicated(
    ["bankrupt_company_id", "bankrupt_prediction_date", "match_rank"]
).any():
    raise ValueError("Repeated matching rank")
if not matching["size_basis"].isin(["market_cap", "total_assets"]).all():
    raise ValueError("Unknown matching size basis")
for column, expected in [
    (
        "sector_match",
        matching["bankrupt_sector"]
        .str.casefold()
        .eq(matching["control_sector"].str.casefold()),
    ),
    (
        "industry_match",
        matching["bankrupt_industry"]
        .str.casefold()
        .eq(matching["control_industry"].str.casefold()),
    ),
    ("year_match", matching["prediction_year"].eq(matching["control_prediction_year"])),
]:
    if not matching[column].astype(bool).eq(expected).all():
        raise ValueError(f"Inconsistent matching flag: {column}")
if (
    matching["control_reuse_count"].lt(1).any()
    or matching["control_reuse_count"].gt(3).any()
):
    raise ValueError("Invalid matching control reuse count")
size_fields = ["bankrupt_size", "control_size", "size_distance"]
if not np.isfinite(matching[size_fields].to_numpy(dtype=float)).all():
    raise ValueError("Nonfinite matching size metadata")
if not np.allclose(
    matching["size_distance"],
    (matching["bankrupt_size"] - matching["control_size"]).abs(),
    rtol=1e-12,
    atol=1e-12,
):
    raise ValueError("Incorrect matching size distances")
if (
    not matching["bankrupt_size_date"].lt(matching["bankrupt_prediction_date"]).all()
    or not matching["control_size_date"].lt(matching["control_pseudo_event_date"]).all()
):
    raise ValueError("Matching size observations are not before prediction")
if (
    not matching["prediction_year"]
    .eq(matching["bankrupt_prediction_date"].dt.year)
    .all()
    or not matching["control_prediction_year"]
    .eq(matching["control_pseudo_event_date"].dt.year)
    .all()
):
    raise ValueError("Matching prediction years are inconsistent")
year_distance = (
    matching["prediction_year"] - matching["control_prediction_year"]
).abs()
for stage in [1, 2, 3, 4]:
    selected = matching["match_stage"].eq(stage)
    category = "industry" if stage in [1, 3] else "sector"
    same_category = matching[f"bankrupt_{category}"].eq(matching[f"control_{category}"])
    expected_year_distance = 0 if stage in [1, 2] else 1
    if (
        not (same_category & year_distance.eq(expected_year_distance))
        .loc[selected]
        .all()
    ):
        raise ValueError(f"Matching hierarchy mismatch at stage {stage}")

# Confirm roles and classifications using each side's own company/prediction-date key.
events = combined.set_index(KEYS)
retained = {}
for role, id_column, date_column, target in [
    ("bankrupt", "bankrupt_company_id", "bankrupt_prediction_date", 1),
    ("control", "control_company_id", "control_pseudo_event_date", 0),
]:
    side_index = pd.MultiIndex.from_arrays(
        [matching[id_column], matching[date_column]], names=KEYS
    )
    side = events.reindex(side_index).reset_index(drop=True)
    present = side["event_id"].notna()
    retained[role] = present
    if not side.loc[present, "target"].eq(target).all():
        raise ValueError(f"Incorrect {role} target in matching metadata")
    for category in ["sector", "industry"]:
        if (
            not side.loc[present, category]
            .eq(matching.loc[present, f"{role}_{category}"])
            .all()
        ):
            raise ValueError(f"Conflicting {role} Bloomberg {category}")
matching["bankrupt_event_retained"] = retained["bankrupt"]
matching["control_event_retained"] = retained["control"]
matching["pair_retained"] = retained["bankrupt"] & retained["control"]
matching["matching_set_id"] = (
    matching["bankrupt_company_id"]
    + "@"
    + matching["bankrupt_prediction_date"].dt.strftime("%Y-%m-%d")
)
matching["matching_pair_id"] = (
    matching["matching_set_id"]
    + "|"
    + matching["control_company_id"]
    + "@"
    + matching["control_pseudo_event_date"].dt.strftime("%Y-%m-%d")
)
raw_control_reuse = matching.groupby("control_company_id").size()
retained_control_reuse = (
    matching.loc[matching["pair_retained"]].groupby("control_company_id").size()
)
expected_reuse = matching["control_company_id"].map(raw_control_reuse)
if not matching["control_reuse_count"].eq(expected_reuse).all():
    raise ValueError("Control reuse metadata disagrees with primary sample counts")
MAX_CONTROL_REUSE = 3  # Notebook 06's pre-specified reuse limit.
if raw_control_reuse.max() > MAX_CONTROL_REUSE:
    raise ValueError("Primary matches exceed notebook 06's control-company reuse limit")

# JSON preserves each individual pair's size/basis/date, stage, rank, and partner.
# Aggregation uses observed links only; zeros below mean zero retained matching links,
# never missing financial/market observations replaced with zero.
links = {tuple(key): [] for key in combined[KEYS].itertuples(index=False, name=None)}
for row in matching.sort_values(
    ["matching_set_id", "match_rank"], kind="stable"
).to_dict("records"):
    record = {
        c: (
            row[c].strftime("%Y-%m-%d")
            if c in match_dates
            else None
            if pd.isna(row[c])
            else row[c].item()
            if isinstance(row[c], np.generic)
            else row[c]
        )
        for c in row
    }
    for company_column, date_column in [
        ("bankrupt_company_id", "bankrupt_prediction_date"),
        ("control_company_id", "control_pseudo_event_date"),
    ]:
        key = (row[company_column], row[date_column])
        if key in links:
            links[key].append(record.copy())

matching_rows = []
for event in combined[KEYS + ["target", "event_id", "matched_pair_links"]].to_dict(
    "records"
):
    key = (event["company_id"], event["prediction_date"])
    records = links[key]
    if not records or len(records) != event["matched_pair_links"]:
        raise ValueError(
            f"Matching source and feature files are out of sync for {key}: "
            f"the aligned feature files contain {event['matched_pair_links']} matching links, "
            f"but {MATCH_SOURCE} contains {len(records)}. Rerun notebook 07 and feature notebooks "
            "08-12 using the current primary matching sample, then rerun notebook 13."
        )
    is_control = event["target"] == 0
    partner_ids = [
        r["matching_set_id"]
        if is_control
        else r["control_company_id"] + "@" + r["control_pseudo_event_date"]
        for r in records
    ]
    kept_count = sum(r["pair_retained"] for r in records)
    matching_rows.append(
        {
            "company_id": key[0],
            "prediction_date": key[1],
            "metadata_matching_ratio_requested": MATCH_RATIO,
            "metadata_matching_pair_count_original": len(records),
            "metadata_matching_pair_count_retained": kept_count,
            "metadata_matching_pair_count_excluded": len(records) - kept_count,
            "metadata_matching_partner_event_ids_json": json.dumps(
                sorted(set(partner_ids))
            ),
            "metadata_matching_set_ids_json": json.dumps(
                sorted({r["matching_set_id"] for r in records})
            ),
            "metadata_matching_stage_counts_json": json.dumps(
                {
                    str(stage): sum(r["match_stage"] == stage for r in records)
                    for stage in [1, 2, 3, 4]
                }
            ),
            "metadata_matching_size_bases_json": json.dumps(
                sorted({r["size_basis"] for r in records})
            ),
            "metadata_matching_records_json": json.dumps(
                records, allow_nan=False, separators=(",", ":")
            ),
            "metadata_control_company_reuse_original": raw_control_reuse.loc[key[0]]
            if is_control
            else np.nan,
            "metadata_control_company_reuse_retained": retained_control_reuse.get(
                key[0], 0
            )
            if is_control
            else np.nan,
        }
    )
matching_metadata = pd.DataFrame(matching_rows)
matching_metadata["company_id"] = matching_metadata["company_id"].astype("string")
matching_metadata["prediction_date"] = matching_metadata["prediction_date"].astype(
    "datetime64[ns]"
)
if (
    set(matching_metadata.columns)
    .difference(KEYS)
    .intersection(modeling_dataset.columns)
):
    raise ValueError("Matching metadata column collision")
modeling_dataset = modeling_dataset.merge(
    matching_metadata, on=KEYS, how="left", validate="one_to_one", sort=False
)
matching_column_lineage = pd.DataFrame(
    {
        "source_file": MATCH_SOURCE,
        "source_column": "pair records / matching aggregates",
        "output_column": [c for c in matching_metadata if c not in KEYS],
        "group": "metadata",
    }
)
column_lineage = pd.concat([column_lineage, matching_column_lineage], ignore_index=True)

In [5]:
# Validate identity, provenance, missingness, and inherited temporal constraints.
ordered_columns = CORE.copy()
for group in ["metadata", "original", "derived", "trend", "market", "macro", "missing"]:
    ordered_columns.extend(c for c in modeling_dataset if c.startswith(group + "_"))
if len(ordered_columns) != len(modeling_dataset.columns) or len(
    set(ordered_columns)
) != len(ordered_columns):
    raise ValueError("Output contains an ungrouped or duplicate column")
modeling_dataset = modeling_dataset[ordered_columns]
for group in ["original", "derived", "trend", "market", "macro"]:
    columns = [c for c in modeling_dataset if c.startswith(group + "_")]
    if not all(pd.api.types.is_numeric_dtype(modeling_dataset[c]) for c in columns):
        raise ValueError(f"Nonnumeric values in the {group} feature group")
    values = modeling_dataset[columns].to_numpy(dtype=float)
    if np.isinf(values).any():
        raise ValueError(f"Infinite values in the {group} feature group")

# Only source-observation dates are checked here. Filing/outcome dates are future
# labels/metadata; cutoff and window dates retain their upstream definitions.
source_date_columns = [
    c for c in feature_groups["metadata"] if "_date" in c and c != "outcome_end_date"
]
for column in dict.fromkeys(source_date_columns):
    observed = pd.to_datetime(combined[column], errors="raise")
    present = observed.notna()
    if not observed.loc[present].le(combined.loc[present, "prediction_date"]).all():
        raise ValueError(f"Future source date in {column}")
financial_date = pd.to_datetime(combined["financial_reference_date"], errors="raise")
positive = modeling_dataset["target"].eq(1)

# Decode JSON once to verify all fields and both members' retained link multiplicity.
record_occurrences = {}
for row in modeling_dataset[
    [
        "metadata_matching_records_json",
        "metadata_matching_pair_count_original",
        "metadata_matching_pair_count_retained",
    ]
].itertuples(index=False, name=None):
    records = json.loads(row[0])
    for record in records:
        pair_id = record["matching_pair_id"]
        record_occurrences[pair_id] = record_occurrences.get(pair_id, 0) + 1
for pair in matching.to_dict("records"):
    expected_occurrences = int(pair["bankrupt_event_retained"]) + int(
        pair["control_event_retained"]
    )

In [6]:
# Report the final groups and remaining matched-set sizes before exporting.
group_summary = column_lineage.groupby("group", sort=False).agg(
    columns=("output_column", "size")
)
control_usage_report = (
    pd.DataFrame(
        {
            "original_pair_uses": raw_control_reuse,
            "retained_pair_uses": retained_control_reuse,
        }
    )
    .fillna(0)
    .astype(int)
)
control_usage_report.index.name = "control_company_id"
feature_output_columns = [
    rename_map[c]
    for group in ["original", "derived", "trend", "market", "macro"]
    for c in feature_groups[group]
]

output_path = PROCESSED / "modeling_dataset_unprocessed.csv"
modeling_dataset.to_csv(output_path, index=False, date_format="%Y-%m-%d")
compact_columns_present = [
    column for column in COMPACT_FEATURE_COLUMNS if column in modeling_dataset.columns
]
reports_dir = project_paths(ROOT).reports / "04_feature_diagnostics"
reports_dir.mkdir(parents=True, exist_ok=True)
feature_specifications = {
    "compact": {
        "columns": compact_columns_present,
        "unavailable_columns": [
            column
            for column in COMPACT_FEATURE_COLUMNS
            if column not in compact_columns_present
        ],
        "all_missing_columns": [
            column
            for column in compact_columns_present
            if modeling_dataset[column].isna().all()
        ],
        "selection_rule": "Pre-specified; compare using training and validation only.",
    },
    "full": {
        "columns": [
            column
            for group in ["original", "derived", "trend", "market", "macro", "missing"]
            for column in modeling_dataset
            if column.startswith(group + "_")
        ],
        "selection_rule": "Full feature family; compare using training and validation only.",
    },
}
feature_specifications["compact"]["notes"] = [
    "macro_credit_spread is missing because the configured Bloomberg series is a corporate bond total-return index, not a credit spread."
]
with (reports_dir / "feature_specifications.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(feature_specifications, handle, indent=2)